In [1]:
import json
from bs4 import BeautifulSoup

file = open("../../data/course-info.jsonlines", "r")
lines = file.readlines()

reqs = []

for line in lines:
    course = json.loads(line)

    prereq = course.get("prereq")
    antireq = course.get("antireq")
    coreq = course.get("coreq")

    soup = BeautifulSoup(str(prereq), 'html.parser')
    prereq = str(soup.get_text())

    soup = BeautifulSoup(str(antireq), 'html.parser')
    antireq = str(soup.get_text())
    
    soup = BeautifulSoup(str(coreq), 'html.parser')
    coreq = str(soup.get_text())

    if (prereq != "None"):
        reqs.append(prereq)
    
    if(coreq != "None"):
        reqs.append(coreq)

    if(antireq != "None"):
        reqs.append(antireq)

print("Reqs:", len(reqs))


Reqs: 4759


In [4]:
import pandas as pd
import json

dictionary = {
    "FACULTY": "faculty",
    "DEPARTMENT": "department",
    "PROGRAM": "program",
    "COURSE_TITLE": "course-title",
}

entity_types = []

def sort(e):
    return len(e[0])

for entity_type in dictionary:
    file_name = dictionary[entity_type]
    file = open(f"../../data/{file_name}.jsonlines", "r")
    lines = file.readlines()

    for line in lines:
        title_ = json.loads(line).get("title")

        if(entity_type == "FACULTY" and "of" not in title_):
            title = "Faculty of " + title_
        elif(entity_type == "DEPARTMENT"):
            title = "Department of " + title_
        elif(entity_type == "PROGRAM"):
            title = title_ + " Porgram"
        else:
            title = title_

        entity_types.append([title, entity_type])
    
    file.close()


# Exceptions
entity_types.append(["Department of Electrical and Software Engineering", "DEPARTMENT"])


entity_types = sorted(entity_types, key=sort, reverse=True)
entity_types = pd.Series(entity_types).drop_duplicates().tolist()

file = open("entity_types.jsonlines", "w")
for entity_type in entity_types:
    line = json.dumps(entity_type) + "\n"
    file.write(line)
file.close()

pd.DataFrame(entity_types, columns=["Title", "Type"])



,Title,Type
0,"Department of School of Languages, Linguistics...",DEPARTMENT
1,"Department of Microbiology, Immunology & Infec...",DEPARTMENT
2,Department of Community Rehabilitation & Disab...,DEPARTMENT
3,Department of Mechanical and Manufacturing Eng...,DEPARTMENT
4,Department of Biomedical Engineering Graduate ...,DEPARTMENT
...,...,...
327,Music,COURSE_TITLE
328,Arts,COURSE_TITLE
329,Film,COURSE_TITLE
330,Art,COURSE_TITLE


In [28]:
import re

file = open("testing_data.jsonlines", "w")

for req in reqs:
    ents = []
    req_ = req
    
    for entity_type in entity_types:
        title = entity_type[0]
        type_name = entity_type[1]

        pattern = rf"\b({title})\b"
        regex = re.finditer(pattern, req_)

        if(not regex):
            continue
        
        for group in regex:
            entity = [int(group.start()), int(group.end()), type_name]
            ents.append(entity)
            req_ = req_[:group.start()] + (' ' * (group.end() - group.start())) + req_[group.end():]

    if(ents == []):
        continue

    line = [req, {"entities": ents}]
    line = json.dumps(line) + "\n"
    file.write(line)

file.close()
